# Análise exploratória do datamart de risco (Doença de Chagas aguda)

Ponto de partida para a análise exploratória do TCC. Carrega `data/datamart/` (gerado por `run_etl.py`/`uv run tcc-datasus etl`) e os relatórios de avaliação já produzidos por `run_train.py`/`uv run tcc-datasus train` -- não recalcula nada, só visualiza o que o pipeline já gera.

Pré-requisito: `data/datamart/` precisa existir (`uv run tcc-datasus etl`).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import polars as pl
import seaborn as sns

from tcc_datasus.train import load_datamart

sns.set_theme(style="whitegrid")

DATAMART_DIR = Path("../data/datamart")
REPORTS_DIR = Path("../reports")

datamart = load_datamart(DATAMART_DIR)
datamart.head()

## Visão geral: quantos município-anos, quantos de risco, por UF

In [ ]:
resumo_por_uf = datamart.group_by("uf").agg(
    pl.len().alias("municipio_anos"),
    pl.col("risco").sum().alias("municipio_anos_de_risco"),
    pl.col("risco").mean().alias("proporcao_de_risco"),
).sort("uf")
resumo_por_uf

## Casos confirmados ao longo do tempo, por UF

In [ ]:
casos_por_ano = (
    datamart.group_by(["uf", "ano"])
    .agg(pl.col("casos").sum().alias("casos_confirmados"))
    .sort(["uf", "ano"])
    .to_pandas()
)

fig, ax = plt.subplots(figsize=(10, 5))
sns.lineplot(data=casos_por_ano, x="ano", y="casos_confirmados", hue="uf", marker="o", ax=ax)
ax.set_title("Casos confirmados de Chagas aguda por ano e UF")
ax.set_xlabel("Ano")
ax.set_ylabel("Casos confirmados")
plt.show()

## Distribuição da população (por UF) nos município-anos de risco vs. sem risco

`populacao` vem de `fetch_ibge_population` (API do SIDRA/IBGE), que só tem dados em nível de UF -- o SINAN não traz população por município. Cada município de uma UF compartilha o mesmo valor (o da UF inteira), então o gráfico abaixo compara UFs, não municípios individualmente.

In [ ]:
if "populacao" in datamart.columns:
    fig, ax = plt.subplots(figsize=(8, 5))
    sns.boxplot(data=datamart.to_pandas(), x="risco", y="populacao", ax=ax)
    ax.set_title("População da UF (aproximação do SINAN), por rótulo de risco")
    ax.set_xlabel("Município-ano de risco (0/1)")
    ax.set_ylabel("População da UF")
    ax.set_yscale("log")
    plt.show()
else:
    print("Datamart sem a coluna 'populacao' -- rode 'uv run tcc-datasus etl' novamente para cruzar com o IBGE.")

## Resultados já produzidos por `run_train.py` / `uv run tcc-datasus train`

Os relatórios abaixo já existem em `reports/` depois de uma rodada de treino -- aqui só lemos o que já foi calculado, sem reprocessar nada.

In [ ]:
resumo_path = REPORTS_DIR / "resumo_avaliacao.md"
if resumo_path.exists():
    print(resumo_path.read_text(encoding="utf-8"))
else:
    print("reports/resumo_avaliacao.md ainda não existe -- rode 'uv run tcc-datasus train' primeiro.")